# MetaJudge: What Can We Learn About How LLMs Monitor and Control Their Own Cognition?

**Competition:** Kaggle — Measuring Progress Toward AGI (Metacognition Track)  
**Benchmark version:** v0.5.5.1  
**Models evaluated:** Gemini 2.5 Flash, Gemini 2.5 Pro, Claude Sonnet 4, Claude Haiku 4.5, DeepSeek V3.1  

---

## Why Metacognition Matters

Current AI benchmarks measure what models *know* — can they answer questions, solve math problems, write code? But they rarely measure what models *know about their own knowledge*. A model that gives a confident wrong answer is more dangerous than one that says "I'm not sure." A model that blindly answers when it should ask for clarification is less useful than one that recognizes ambiguity.

MetaJudge measures **metacognitive monitoring** (does the model know what it knows?) and **metacognitive control** (does it act appropriately on that knowledge?). These are distinct abilities: a model can be well-calibrated but still fail to abstain when it should, or vice versa.

## What MetaJudge Measures

| Family | Axis | What It Tests | Score |
|--------|------|---------------|-------|
| **A: Calibration** | Monitoring | Is the model's confidence aligned with its actual accuracy? | 1 − Brier (strictly proper) |
| **B: Selective Abstention** | Control | Does the model answer, clarify, verify, or abstain appropriately? | UWAA (utility-weighted) |
| **Bridge** | Coupling | Does monitoring quality predict control quality? | Spearman ρ |

The benchmark deliberately probes failure modes: monitoring traps, anchoring effects, ambiguity metacognition, prototype errors, and more.

In [ ]:
# Cell 1 — Imports & Setup
import sys, os, json, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', category=FutureWarning)

# Package path for Kaggle
_pkg_paths = [
    "/kaggle/input/datasets/seanmcgee2025/metajudge-benchmark",
    "/kaggle/input/metajudge-benchmark",
]
for _p in _pkg_paths:
    if os.path.exists(_p):
        sys.path.insert(0, _p)
        break

from metajudge.notebook_helpers import (
    resolve_data_root, resolve_output_dir,
    load_benchmark_dataset, load_family_b_dataset,
    load_clean_manifest, get_excluded_item_ids, filter_clean_subset,
    short_model_name, SWEEP_MODELS,
    format_leaderboard_cal, format_leaderboard_fb,
)
from metajudge.scoring.calibration_metrics import (
    expected_calibration_error, overconfidence_rate,
    accuracy_by_confidence_bucket,
)
from metajudge.scoring.abstention_metrics import (
    compute_uwaa, compute_action_f1,
)
from metajudge.scoring.statistics import (
    mcnemar_test, paired_permutation_test, paired_bootstrap_ci,
    spearman_with_ci, holm_correction,
)

print("MetaJudge narrative notebook loaded.")
print(f"Sweep models: {len(SWEEP_MODELS)}")

In [ ]:
# Cell 2 — Load Data & Apply Clean Subset
import csv
from collections import defaultdict

data_root = resolve_data_root()
output_dir = resolve_output_dir()

# Load benchmark items
cal_items = load_benchmark_dataset(data_root)
fb_items = load_family_b_dataset(data_root)
manifest = load_clean_manifest(data_root)
cal_excluded, fb_excluded = get_excluded_item_ids(manifest)

cal_clean = filter_clean_subset(cal_items, cal_excluded)
fb_clean = filter_clean_subset(fb_items, fb_excluded)

print(f"Calibration: {len(cal_items)} total → {len(cal_clean)} clean "
      f"({len(cal_excluded)} excluded)")
print(f"Family B: {len(fb_items)} total → {len(fb_clean)} clean "
      f"({len(fb_excluded)} excluded)")
print(f"\nExclusion criteria:")
print(f"  Cal: {manifest['exclusion_criteria']['calibration']}")
print(f"  FB:  {manifest['exclusion_criteria']['family_b']}")

In [ ]:
# Cell 3 — Load Pre-computed v0.5.5.1 Results
# (Results from the 5-model sweep are loaded from CSV rather than
# re-running the full sweep, which costs ~$50 in API calls.)

# Try multiple data locations
_result_dirs = [
    "/tmp/v0551/v0.5.5.1 - results",
    os.path.join(data_root, "results"),
    "outputs",
]
result_dir = next((d for d in _result_dirs if os.path.exists(d)), None)

if result_dir:
    # Find the calibration CSV
    cal_csv = None
    for f in os.listdir(result_dir):
        if f.startswith("calibration_item_audit") and f.endswith(".csv"):
            cal_csv = os.path.join(result_dir, f)
        elif f.startswith("family_b_item_audit") and f.endswith(".csv"):
            fb_csv = os.path.join(result_dir, f)

# Load calibration results
cal_results = defaultdict(dict)  # {model: {item_id: {...}}}
with open(cal_csv) as f:
    for row in csv.DictReader(f):
        m = row["model_name"]
        iid = row["item_id"]
        cal_results[m][iid] = {
            "is_correct": row["is_correct"] == "True",
            "brier_score": float(row["brier_score"]),
            "confidence": float(row["confidence"]),
            "mechanism": row["mechanism"],
        }

# Load Family B results
fb_results = defaultdict(dict)
with open(fb_csv) as f:
    for row in csv.DictReader(f):
        m = row["model_name"]
        iid = row["item_id"]
        fb_results[m][iid] = {
            "model_decision": row["model_decision"],
            "gold_action": row["gold_action"],
            "utility": float(row["utility"]),
            "confidence": float(row["confidence"]),
            "is_correct": row["is_correct"] == "True",
        }

# Filter to clean subsets
cal_clean_ids = {item["item_id"] for item in cal_clean}
fb_clean_ids = {item["item_id"] for item in fb_clean}

cal_results_clean = {
    m: {iid: v for iid, v in items.items() if iid in cal_clean_ids}
    for m, items in cal_results.items()
}
fb_results_clean = {
    m: {iid: v for iid, v in items.items() if iid in fb_clean_ids}
    for m, items in fb_results.items()
}

print(f"Loaded results for {len(cal_results)} models")
for m in sorted(cal_results.keys()):
    nc = len(cal_results_clean.get(m, {}))
    nf = len(fb_results_clean.get(m, {}))
    print(f"  {short_model_name(m):15s}: {nc} cal (clean), {nf} FB (clean)")

## Dataset Cleaning

We report results on a **clean subset** that excludes items identified as potentially problematic in the QA audit:

- **Calibration**: 12 items excluded where ≥3 of 5 models answered incorrectly (suggesting item-level problems, not individual model failures)
- **Family B**: 12 items excluded where models systematically answered directly instead of verifying (suggesting ambiguous action boundaries)

All 10 calibration mechanisms and all 4 Family B actions are preserved. Full-set results are available as a sensitivity check in the appendix.

---

## Calibration Results (Clean Set)

In [ ]:
# Cell 4 — Calibration Leaderboard

cal_metrics = {}
for model, items in cal_results_clean.items():
    confs = [v["confidence"] for v in items.values()]
    corrects = [v["is_correct"] for v in items.values()]
    briers = [v["brier_score"] for v in items.values()]
    cal_metrics[model] = {
        "accuracy": float(np.mean(corrects)),
        "mean_brier": float(np.mean(briers)),
        "ece": expected_calibration_error(confs, corrects),
        "overconfidence_rate": overconfidence_rate(confs, corrects),
        "n_items": len(items),
    }

leaderboard = format_leaderboard_cal(cal_metrics)
df_lb = pd.DataFrame(leaderboard)
print("\n=== Calibration Leaderboard (Clean Set, 105 items) ===")
print(df_lb.to_string(index=False))

In [ ]:
# Cell 5 — Calibration Reliability Diagram
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.1)
colors = sns.color_palette("tab10", n_colors=len(cal_results_clean))
model_order = sorted(cal_results_clean.keys(), key=short_model_name)
model_colors = {m: colors[i] for i, m in enumerate(model_order)}

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Perfect calibration")

for model in model_order:
    items = cal_results_clean[model]
    confs = [v["confidence"] for v in items.values()]
    corrects = [v["is_correct"] for v in items.values()]
    bins = accuracy_by_confidence_bucket(confs, corrects)
    xs = [b[0] for b in bins if b[2] > 0]
    ys = [b[1] for b in bins if b[2] > 0]
    ax.plot(xs, ys, "o-", color=model_colors[model],
            label=short_model_name(model), markersize=6)

ax.set_xlabel("Stated Confidence")
ax.set_ylabel("Observed Accuracy")
ax.set_title("Calibration Reliability Diagram (Clean Set)")
ax.legend(loc="lower right", fontsize=9)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
fig.tight_layout()
plt.show()

A well-calibrated model traces the diagonal: when it says it's 80% confident, it should be correct ~80% of the time. Deviations above the diagonal indicate underconfidence; below indicates overconfidence.

---

## Family B: Selective Abstention Results (Clean Set)

Family B tests whether models can choose the *right action* — not just give the right answer. For each item, the model must decide whether to **answer**, **clarify**, **verify**, or **abstain**. The utility matrix rewards appropriate action selection and penalizes overconfident answering.

In [ ]:
# Cell 6 — Family B Leaderboard

fb_metrics = {}
for model, items in fb_results_clean.items():
    if len(items) < 10:  # Skip models with insufficient data
        continue
    utils = [v["utility"] for v in items.values()]
    decisions = [v["model_decision"] for v in items.values()]
    golds = [v["gold_action"] for v in items.values()]
    action_acc = float(np.mean([d == g for d, g in zip(decisions, golds)]))
    
    fb_metrics[model] = {
        "uwaa": compute_uwaa(utils),
        "mean_utility": float(np.mean(utils)),
        "action_accuracy": action_acc,
        "n_items": len(items),
    }

fb_lb = format_leaderboard_fb(fb_metrics)
df_fb = pd.DataFrame(fb_lb)
print("\n=== Family B Leaderboard (Clean Set) ===")
print(df_fb.to_string(index=False))

# Note Gemini Flash gap
flash_n = len(fb_results_clean.get("google/gemini-2.5-flash", {}))
if flash_n < 10:
    print(f"\nNote: Gemini Flash has only {flash_n} Family B item(s) "
          f"and is excluded from pairwise comparisons.")

In [ ]:
# Cell 7 — Action Distribution Plot

actions = ["answer", "clarify", "verify", "abstain"]
action_colors = {"answer": "#4CAF50", "clarify": "#2196F3",
                 "verify": "#FF9800", "abstain": "#F44336"}

fb_model_order = sorted(fb_metrics.keys(), key=short_model_name)

fig, ax = plt.subplots(figsize=(10, 6))
bottom = np.zeros(len(fb_model_order))
for action in actions:
    vals = []
    for m in fb_model_order:
        items = fb_results_clean[m]
        decisions = [v["model_decision"] for v in items.values()]
        vals.append(sum(1 for d in decisions if d == action) / len(decisions))
    ax.bar([short_model_name(m) for m in fb_model_order], vals,
           bottom=bottom, label=action, color=action_colors[action])
    bottom += np.array(vals)

ax.set_ylabel("Proportion")
ax.set_title("Action Distribution by Model (Family B, Clean Set)")
ax.legend()
fig.tight_layout()
plt.show()

---

## Pairwise Statistical Comparisons

We use non-parametric paired tests (same items, different models) with Holm-Bonferroni correction for multiple comparisons. All tests use 10,000 permutations/resamples with seed=42.

In [ ]:
# Cell 8 — Key Pairwise Comparisons (Calibration)
from itertools import combinations

models = sorted(cal_results_clean.keys())
common_items = sorted(set.intersection(
    *(set(cal_results_clean[m].keys()) for m in models)
))

pairwise_results = []
all_p_values = {}

for m_a, m_b in combinations(models, 2):
    pair = f"{short_model_name(m_a)} vs {short_model_name(m_b)}"
    correct_a = [cal_results_clean[m_a][i]["is_correct"] for i in common_items]
    correct_b = [cal_results_clean[m_b][i]["is_correct"] for i in common_items]
    brier_a = [cal_results_clean[m_a][i]["brier_score"] for i in common_items]
    brier_b = [cal_results_clean[m_b][i]["brier_score"] for i in common_items]
    
    mcn = mcnemar_test(correct_a, correct_b)
    perm = paired_permutation_test(brier_a, brier_b)
    boot = paired_bootstrap_ci(brier_a, brier_b)
    
    pairwise_results.append({
        "Pair": pair,
        "Acc A": f"{np.mean(correct_a):.3f}",
        "Acc B": f"{np.mean(correct_b):.3f}",
        "McNemar p": f"{mcn['p_value']:.4f}",
        "Brier Δ": f"{boot['observed_diff']:.4f}",
        "Brier 95% CI": f"[{boot['ci_lower']:.4f}, {boot['ci_upper']:.4f}]",
    })
    all_p_values[f"{pair}_acc"] = mcn["p_value"]
    all_p_values[f"{pair}_brier"] = perm["p_value"]

# Holm-Bonferroni correction
corrected = holm_correction(all_p_values)
n_sig = sum(1 for v in corrected.values() if v["significant_0.05"])

print(f"\n=== Calibration Pairwise Comparisons (Clean Set, n={len(common_items)}) ===")
df_pw = pd.DataFrame(pairwise_results)
print(df_pw.to_string(index=False))
print(f"\nHolm-Bonferroni: {n_sig}/{len(corrected)} tests significant at α=0.05")

In [ ]:
# Cell 9 — Brier Score Forest Plot

fig, ax = plt.subplots(figsize=(10, 5))
pairs = [r["Pair"] for r in pairwise_results]

for i, (m_a, m_b) in enumerate(combinations(models, 2)):
    brier_a = [cal_results_clean[m_a][iid]["brier_score"] for iid in common_items]
    brier_b = [cal_results_clean[m_b][iid]["brier_score"] for iid in common_items]
    boot = paired_bootstrap_ci(brier_a, brier_b)
    ax.errorbar(boot["observed_diff"], i,
                xerr=[[boot["observed_diff"] - boot["ci_lower"]],
                      [boot["ci_upper"] - boot["observed_diff"]]],
                fmt="o", color="steelblue", capsize=4, markersize=6)

ax.axvline(0, color="red", linestyle="--", alpha=0.4)
ax.set_yticks(range(len(pairs)))
ax.set_yticklabels(pairs, fontsize=9)
ax.set_xlabel("Brier Score Difference (A − B)")
ax.set_title("Pairwise Brier Differences with 95% Bootstrap CI (Clean Set)")
fig.tight_layout()
plt.show()

---

## Bridge Analysis: Does Monitoring Quality Predict Control Quality?

The bridge analysis asks: if a model is well-calibrated (good monitoring), does it also make better action decisions (good control)? A strong positive Spearman correlation between confidence quality and control quality would suggest these are coupled abilities. A weak or absent correlation would suggest they are dissociable — and therefore worth measuring separately.

In [ ]:
# Cell 10 — Bridge: Confidence-Accuracy Correlation

bridge_results = []
for model in model_order:
    items = cal_results_clean[model]
    confs = [v["confidence"] for v in items.values()]
    corrects = [float(v["is_correct"]) for v in items.values()]
    sp = spearman_with_ci(confs, corrects)
    bridge_results.append({
        "Model": short_model_name(model),
        "Spearman ρ": f"{sp['rho']:.3f}",
        "p-value": f"{sp['p_value']:.4f}",
        "95% CI": f"[{sp['ci_lower']:.3f}, {sp['ci_upper']:.3f}]",
    })

print("\n=== Confidence-Accuracy Correlation (Clean Set) ===")
df_bridge = pd.DataFrame(bridge_results)
print(df_bridge.to_string(index=False))

In [ ]:
# Cell 11 — Confidence Distribution Violin Plot

conf_data = []
for model in model_order:
    for v in cal_results_clean[model].values():
        conf_data.append({
            "Model": short_model_name(model),
            "Confidence": v["confidence"],
        })

df_conf = pd.DataFrame(conf_data)

fig, ax = plt.subplots(figsize=(10, 5))
sns.violinplot(data=df_conf, x="Model", y="Confidence", hue="Model",
               ax=ax, palette="tab10", inner="quartile", legend=False)
ax.set_title("Confidence Distribution by Model (Clean Set)")
ax.set_ylim(0, 1.05)
fig.tight_layout()
plt.show()

---

## Per-Mechanism Analysis

MetaJudge uses 10 distinct calibration mechanisms, each designed to probe a different failure mode. Some mechanisms are easy for all models; others reveal systematic weaknesses.

In [ ]:
# Cell 12 — Per-Mechanism Accuracy Heatmap

# Compute per-mechanism accuracy
mechanisms = sorted(set(
    v["mechanism"] for items in cal_results_clean.values()
    for v in items.values()
))

mech_data = []
for mech in mechanisms:
    row = {"Mechanism": mech}
    for model in model_order:
        items = {k: v for k, v in cal_results_clean[model].items()
                 if v["mechanism"] == mech}
        if items:
            row[short_model_name(model)] = float(np.mean(
                [v["is_correct"] for v in items.values()]))
        else:
            row[short_model_name(model)] = float("nan")
    row["n"] = sum(1 for v in cal_results_clean[model_order[0]].values()
                   if v["mechanism"] == mech)
    mech_data.append(row)

df_mech = pd.DataFrame(mech_data).set_index("Mechanism")
n_col = df_mech.pop("n")

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(df_mech, annot=True, fmt=".2f", cmap="RdYlGn",
            vmin=0, vmax=1, linewidths=0.5, ax=ax)
ax.set_title("Accuracy by Mechanism × Model (Clean Set)")
fig.tight_layout()
plt.show()

# Show item counts
print("\nItems per mechanism:", dict(zip(mechanisms, n_col.values)))

---

## What This Benchmark Reveals

MetaJudge shows that metacognitive quality varies substantially across frontier models, and that **monitoring and control are partially dissociable**:

1. **Calibration quality differs between models** — some models are systematically overconfident on wrong answers, while others maintain more realistic confidence estimates.

2. **Action selection quality is distinct from answer quality** — a model that scores well on calibration doesn't necessarily choose the right metacognitive action (answer vs. abstain vs. verify vs. clarify).

3. **Mechanism-level analysis reveals systematic failure patterns** — models don't fail uniformly; they have characteristic weaknesses (e.g., anchoring traps, monitoring traps) that a global score would obscure.

4. **The benchmark is robust to item cleaning** — excluding the 24 suspect items preserves the main pattern of results while making the interpretation cleaner.

These findings suggest that metacognitive ability deserves its own benchmark dimension, separate from knowledge and reasoning benchmarks. A model can ace a knowledge test while being dangerously overconfident about its wrong answers — and MetaJudge is designed to measure exactly that gap.

---

## Limitations and Next Steps

- **Sample sizes**: 105 calibration items and 72 Family B items provide adequate power for detecting moderate effects but limit per-mechanism statistical inference.
- **RLHF mechanism**: Only 2 clean items remain — results for this mechanism are exploratory only.
- **Gemini Flash Family B**: Only 1 item available; excluded from pairwise comparisons.
- **Temporal sensitivity**: Some items reference time-dependent facts; these are flagged in the audit.
- **Future work**: Families C-E (self-correction, grounding sensitivity, control policy adaptation) are designed but not yet instrumented.

---

*Full statistical details, including all p-values, effect sizes, and sensitivity analyses, are available in the [stats report](outputs/metajudge_v0551_stats_report.docx) and [theoretical backgrounder](outputs/metajudge_v0551_stats_backgrounder.md).*